# EDA Workflow — Systematic Exploratory Data Analysis

A systematic EDA template you can apply to any dataset. Follow this checklist and you'll never miss something important.

**EDA Stages:**
1. Data Loading & Basic Info
2. Missing Value Analysis
3. Duplicate Detection
4. Distribution Analysis
5. Categorical Analysis
6. Correlation Analysis
7. Outlier Detection
8. Target Variable Analysis
9. Key Questions Summary

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# Generate realistic dataset with issues (as in real life)
n = 1500
df = pd.DataFrame({
    'age':          np.random.randint(18, 80, n),
    'income':       np.random.lognormal(10.5, 0.8, n),
    'credit_score': np.random.normal(680, 100, n).clip(300, 850),
    'loan_amount':  np.random.lognormal(10, 0.5, n),
    'employment':   np.random.choice(['employed', 'self_employed', 'unemployed', 'retired'], n, p=[0.6,0.2,0.1,0.1]),
    'education':    np.random.choice(['high_school', 'bachelor', 'master', 'phd'], n, p=[0.3,0.4,0.2,0.1]),
    'defaulted':    np.random.choice([0, 1], n, p=[0.8, 0.2]),  # target
})

# Inject missing values
df.loc[np.random.choice(n, 120, replace=False), 'income'] = np.nan
df.loc[np.random.choice(n, 80, replace=False), 'credit_score'] = np.nan
df.loc[np.random.choice(n, 30, replace=False), 'education'] = np.nan

# Inject duplicates
df = pd.concat([df, df.iloc[:15]], ignore_index=True)

# Inject outliers
df.loc[[0,1,2], 'income'] = [10_000_000, 8_000_000, 12_000_000]

print(f'Dataset shape: {df.shape}')

## Stage 1: Basic Info

In [ ]:
print('=== BASIC INFO ===')
print(f'Rows: {df.shape[0]:,}, Columns: {df.shape[1]}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print()
print('Column info:')
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.count(),
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(1),
    'nunique': df.nunique(),
})
print(info_df)
print()
print('Numeric summary:')
df.describe().round(2)

## Stage 2 & 3: Missing Values & Duplicates

In [ ]:
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('pct', ascending=False)
print(missing_df)

# Visualize missing patterns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
missing_df['pct'].plot(kind='bar', ax=ax1, color='red', alpha=0.7)
ax1.set_title('Missing Value % by Column'); ax1.set_ylabel('Missing %')
ax1.tick_params(axis='x', rotation=45)

# Missing pattern heatmap
mask_data = df[missing_df.index].isnull().astype(int)
ax2.imshow(mask_data.T, aspect='auto', cmap='RdBu', interpolation='none')
ax2.set_yticks(range(len(missing_df)))
ax2.set_yticklabels(missing_df.index)
ax2.set_title('Missing Value Pattern (red=missing)')
ax2.set_xlabel('Row index')

plt.tight_layout(); plt.show()

print('\n=== DUPLICATES ===')
n_dups = df.duplicated().sum()
print(f'Duplicate rows: {n_dups} ({n_dups/len(df)*100:.1f}%)')
if n_dups > 0:
    print('Action: df = df.drop_duplicates()')
    df = df.drop_duplicates()
    print(f'After dedup: {len(df)} rows')

## Stage 4-6: Distributions, Categories & Correlations

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols.remove('defaulted')  # treat target separately

# Distribution for each numeric column
n_num = len(numeric_cols)
fig, axes = plt.subplots(2, n_num, figsize=(4*n_num, 8))

for idx, col in enumerate(numeric_cols):
    data_col = df[col].dropna()
    skew = data_col.skew()
    
    # Histogram
    axes[0, idx].hist(data_col, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
    axes[0, idx].set_title(f'{col}\nskew={skew:.2f}')
    axes[0, idx].set_xlabel(col)
    
    # Box plot
    axes[1, idx].boxplot(data_col, patch_artist=True, 
                          boxprops=dict(facecolor='steelblue', alpha=0.7))
    axes[1, idx].set_title('Box plot')

plt.suptitle('Numeric Feature Distributions', fontsize=13)
plt.tight_layout(); plt.show()

# Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[numeric_cols + ['defaulted']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, ax=ax, square=True)
ax.set_title('Correlation Matrix (lower triangle)')
plt.tight_layout(); plt.show()

## Stage 7-9: Outliers & Target Analysis

In [ ]:
# Target variable analysis — always the most important step!
target = 'defaulted'
print('=== TARGET VARIABLE ANALYSIS ===')
print(f'Target: {target}')
print(df[target].value_counts())
print(f'\nClass balance: {df[target].mean():.1%} positive (defaulted)')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Target distribution
df[target].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'red'], alpha=0.8)
axes[0].set_title('Target Distribution'); axes[0].set_xticklabels(['Not defaulted', 'Defaulted'], rotation=0)

# Feature vs target: income by default status
for val, color, label in [(0, 'steelblue', 'Not defaulted'), (1, 'red', 'Defaulted')]:
    subset = df[df[target] == val]['income'].dropna()
    axes[1].hist(np.log10(subset + 1), bins=30, alpha=0.6, color=color, label=label, density=True)
axes[1].set_title('Income Distribution by Default Status')
axes[1].set_xlabel('log10(Income)'); axes[1].legend()

# Default rate by categorical feature
default_by_emp = df.groupby('employment')[target].mean().sort_values(ascending=False)
default_by_emp.plot(kind='bar', ax=axes[2], color='darkorange', alpha=0.8)
axes[2].set_title('Default Rate by Employment Type')
axes[2].set_ylabel('Default Rate'); axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('Target Variable Deep Dive', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('\n=== EDA SUMMARY & NEXT STEPS ===')
print('1. Missing values: impute income (lognormal → use median), credit_score (mean), education (mode)')
print('2. Outliers: cap income at 99th percentile or apply log transform')
print('3. Class imbalance: 20% positive rate — consider SMOTE or class_weight=balanced')
print('4. Strong features: employment type shows clear difference in default rates')
print('5. Encoding needed: employment, education (ordinal or OHE)')